# calculations.ipynb — CBAM Exposure Estimation: Default Emission Values

Estimates embedded CO2 exposure and CBAM certificate cost for each exporting
country and sector, based on 2024 EU import volumes and EU Commission default
emission values.

**Methodology:**
- Embedded CO2 (tCO2) = import volume (tonnes) x default emission value (tCO2/t)
- Estimated cost (EUR) = embedded_co2 x certificate price (EUR/tCO2)
- Default emission values in `default_2026` already include the applicable markup
  (10% for most sectors, 1% for fertilizers), as confirmed by the source xlsx
  column header 'including mark-up'. No additional markup is applied here.
- Certificate price: EUR 75.36/tCO2 (first official CBAM price, European
  Commission, April 2026). Used only for the convenience sector cost columns
  in `cbam_cost_by_country` — see the warning cell before the write step.
- Trade flow year: 2024

**Route scenarios:**
- `_high_route`: highest default_2026 per (country, cn_code) — upper bound
  cost estimate
- `_low_route`: lowest default_2026 per (country, cn_code) — lower bound
  cost estimate
- Where only one route exists per (country, cn_code), high and low are identical
- `has_route_variation` flags rows where high and low differ

**Join logic:**
- `cbam_defaults` is the left (anchor) table. All 119 countries with published
  defaults are retained regardless of whether they have matching 2024 trade data.
- Countries with no trade flow match receive `import_tonnes = 0`.
- The 116 trade flow countries with no CBAM default are correctly excluded.
- Sector labels derived from CN code prefixes via `SECTOR_MAP`.

**EU vs global export values:**
- `eu_import_value_eur`: value of CBAM-sector goods the EU imported from that
  country (from `trade_flows`). This is an EU-perspective figure only.
- `global_cbam_export_value_eur`: total value of CBAM-sector goods that country
  exports to all markets worldwide (from `global_exports` via Comtrade). This
  is the correct denominator for measuring a country's overall CBAM exposure.
- `cbam_eu_exposure_ratio_high/low`: embedded_co2 / global_cbam_export_value_eur.
  Units: tCO2 per EUR of global CBAM-sector exports. Price-independent.
  Only computed where Comtrade data_year = 2024 (year-consistent with trade_flows).
  NULL for fallback countries. The dashboard multiplies by cert_price at runtime
  to derive cost-as-share-of-exports, and shows no value for NULL countries.
- `export_data_year`: actual Comtrade data year for each country.
- `is_fallback_year`: 1 where Comtrade used a pre-2024 year, 0 otherwise.

**Known limitations:**
- 22 countries with published CBAM defaults had no recorded EU imports in 2024.
- CBAM certificate price is volatile and will change as the market matures.
- The markup in `default_2026` is 2026-specific. For future-year analysis,
  use `default_2027` (20% markup) or `default_2028_onwards` (30% markup).

**Output tables written to db/cbam.db:**
- `cbam_cost_by_country_sector` — grain: (country, sector, cn_code)
- `cbam_cost_by_country` — grain: (country), aggregated across all sectors
- `cbam_cost_by_sector` — grain: (sector), aggregated across all countries

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
# Standard libraries for data manipulation, database access, and path handling.

import sqlite3
import pandas as pd
from pathlib import Path

In [2]:
# ── Display formatting ────────────────────────────────────────────────────────
# Suppress scientific notation globally for this notebook session.

pd.set_option('display.float_format', '{:,.2f}'.format)

In [3]:
# ── Constants ─────────────────────────────────────────────────────────────────
# All key assumptions defined here as named constants.
#
# BASE_PRICE_EUR:
#   First official CBAM certificate price published by the European Commission
#   on 7 April 2026. Tracks the EU ETS and will fluctuate over time.
#   Source: https://www.homaio.com/post/eu-ets-definitions-updated-guide-for-2025
#   Used ONLY for the per-sector convenience cost columns in cbam_cost_by_country.
#   All primary exposure metrics stored in the DB are price-independent.
#   The dashboard recalculates all euro costs at runtime using a slider.
#
# REFERENCE_YEAR:
#   2024 is the most recent complete year and the last full year before
#   CBAM's definitive phase began (1 January 2026).
#
# SECTOR_MAP:
#   Maps CN code prefixes to sector labels. Derived from the defaults table
#   so all 119 countries carry a sector label including zero-trade rows.
#   Longest prefix matched first to avoid shorter keys capturing longer ones.

BASE_PRICE_EUR = 75.36   # EUR/tCO2, first official EC CBAM price, April 2026
REFERENCE_YEAR = 2024    # Trade flow reference year

SECTOR_MAP = {
    '2507': 'Cement',
    '2523': 'Cement',
    '2601': 'Iron and Steel',
    '2716': 'Electricity',
    '2804': 'Hydrogen',
    '2808': 'Fertilizers',
    '2814': 'Fertilizers',
    '2834': 'Fertilizers',
    '3102': 'Fertilizers',
    '3105': 'Fertilizers',
    '6810': 'Cement',
    '6811': 'Cement',
    '72'  : 'Iron and Steel',
    '73'  : 'Iron and Steel',
    '76'  : 'Aluminium',
}

In [4]:
# ── Database connection ───────────────────────────────────────────────────────
# Connect to the SQLite database generated in notebook 08.
# projects/01_country_exposure/ is two levels below the repo root.

DB_PATH = Path('../../db/cbam.db')

assert DB_PATH.exists(), (
    f'Database not found at {DB_PATH.resolve()}.\n'
    f'Run notebook 08 first to generate cbam.db.'
)

con = sqlite3.connect(DB_PATH)
print(f'Connected to: {DB_PATH.resolve()}')

Connected to: /Users/milcahmaryjoseph/Documents/GitHub/cbam-analysis/db/cbam.db


In [5]:
# ── Load CBAM default emission values ────────────────────────────────────────
# Pull all routes per (country, cn_code) — deduplication happens later.
# Rows where default_2026 IS NULL are excluded. One known case: Chile CN 73061100,
# confirmed as a dash in the legally binding regulation (EUR-Lex page 471/2400).

query_defaults = """
    SELECT
        country,
        cn_code,
        production_route_code,
        production_route,
        direct_emissions,
        indirect_emissions,
        total_emissions,
        default_2026
    FROM cbam_defaults
    WHERE default_2026 IS NOT NULL
"""

df_defaults = pd.read_sql(query_defaults, con)
print(f'CBAM default rows loaded: {len(df_defaults):,}')
print(f'Countries with defaults:  {df_defaults["country"].nunique()}')
print(f'Unique CN codes:          {df_defaults["cn_code"].nunique()}')
df_defaults.head()

CBAM default rows loaded: 10,670
Countries with defaults:  119
Unique CN codes:          262


,country,cn_code,production_route_code,production_route,direct_emissions,indirect_emissions,total_emissions,default_2026
0,Albania,25231000,(A),Grey Clinker / Cement,0.87,0.00,0.87,0.96
1,Albania,25232900,NaN,NaN,0.90,0.03,0.93,1.02
2,Albania,25239000,(A),Grey Clinker / Cement,0.86,0.03,0.89,0.98
3,Albania,28080000,NaN,NaN,2.73,0.04,2.76,2.79
4,Albania,28142000,NaN,NaN,0.65,0.03,0.68,0.69


In [6]:
# ── Assign sector labels from CN codes ───────────────────────────────────────
# Sector labels derived here so all 119 countries carry a sector label,
# including those with no trade flow data.
# Longest prefix matched first to avoid ambiguous matches.

def map_sector(cn_code: str) -> str:
    """Return the CBAM sector name for a given CN code string."""
    for prefix in sorted(SECTOR_MAP.keys(), key=len, reverse=True):
        if cn_code.startswith(prefix):
            return SECTOR_MAP[prefix]
    return None

df_defaults['sector'] = df_defaults['cn_code'].apply(map_sector)

unmapped = df_defaults[df_defaults['sector'].isna()]['cn_code'].unique()
if len(unmapped) > 0:
    raise ValueError(
        f'CN codes not matched by SECTOR_MAP: {unmapped}\n'
        f'Add the relevant prefix to SECTOR_MAP in the constants cell.'
    )

print('Sector assignment complete:')
print(df_defaults['sector'].value_counts())

Sector assignment complete:
sector
Iron and Steel    6251
Fertilizers       2389
Aluminium         1608
Cement             329
Hydrogen            93
Name: count, dtype: int64


In [7]:
# ── Build high-route and low-route defaults per (country, cn_code) ────────────
# For each (country, cn_code) combination, extract:
#   - high_route: the row with the highest default_2026 (upper bound cost)
#   - low_route:  the row with the lowest default_2026 (lower bound cost)
#
# Where only one route exists, high and low will be identical.
# production_route_code and production_route are retained for both scenarios
# so the dashboard can display which route drives each bound.

df_high = (
    df_defaults
    .sort_values('default_2026', ascending=False)
    .drop_duplicates(subset=['country', 'cn_code'], keep='first')
    .reset_index(drop=True)
    .rename(columns={
        'default_2026'          : 'default_2026_high_route',
        'production_route_code' : 'production_route_code_high',
        'production_route'      : 'production_route_high',
    })
)

df_low = (
    df_defaults
    .sort_values('default_2026', ascending=True)
    .drop_duplicates(subset=['country', 'cn_code'], keep='first')
    .reset_index(drop=True)
    [['country', 'cn_code', 'default_2026', 'production_route_code', 'production_route']]
    .rename(columns={
        'default_2026'          : 'default_2026_low_route',
        'production_route_code' : 'production_route_code_low',
        'production_route'      : 'production_route_low',
    })
)

# Merge high and low into a single deduplicated defaults table
df_defaults_both = df_high.merge(
    df_low[['country', 'cn_code',
            'default_2026_low_route',
            'production_route_code_low',
            'production_route_low']],
    on=['country', 'cn_code'],
    how='left'
)

# Flag rows where a cheaper route exists for this (country, cn_code)
df_defaults_both['has_route_variation'] = (
    df_defaults_both['default_2026_high_route'] != df_defaults_both['default_2026_low_route']
)

n_varied = df_defaults_both['has_route_variation'].sum()
print(f'Rows before deduplication:    {len(df_defaults):,}')
print(f'Rows after deduplication:     {len(df_defaults_both):,}')
print(f'Rows with route variation:    {n_varied:,}')
print(f'Rows without route variation: {len(df_defaults_both) - n_varied:,}')

Rows before deduplication:    10,670
Rows after deduplication:     10,641
Rows with route variation:    28
Rows without route variation: 10,613


In [8]:
# ── Load 2024 EU import trade flows (tonnes and EU import value) ──────────────
# Pull both QUANTITY_IN_TONNES and VALUE_IN_EUROS for 2024.
#
# Note on naming: the value column here represents what the EU paid to import
# these goods. It is an EU-perspective figure, not total country export value.
# It is stored as eu_import_value_eur throughout to keep this distinction clear.
#
# Both indicators are pivoted into columns so each (country, cn_code) has one row.

query_trade = f"""
    SELECT
        country,
        iso2,
        cn_code,
        indicator,
        value
    FROM trade_flows
    WHERE year = {REFERENCE_YEAR}
      AND indicator IN ('QUANTITY_IN_TONNES', 'VALUE_IN_EUROS')
"""

df_trade_long = pd.read_sql(query_trade, con)

# Pivot indicators into columns
df_trade = (
    df_trade_long
    .pivot_table(
        index   = ['country', 'iso2', 'cn_code'],
        columns = 'indicator',
        values  = 'value',
        aggfunc = 'sum'
    )
    .reset_index()
    .rename(columns={
        'QUANTITY_IN_TONNES': 'import_tonnes',
        'VALUE_IN_EUROS'    : 'eu_import_value_eur'   # renamed: EU import value only
    })
)
df_trade.columns.name = None

print(f'Trade flow rows after pivot: {len(df_trade):,}')
print(f'Partner countries:           {df_trade["country"].nunique()}')
df_trade.head()

Trade flow rows after pivot: 15,734
Partner countries:           229


,country,iso2,cn_code,import_tonnes,eu_import_value_eur
0,Afghanistan,AF,25070080,0.10,54.00
1,Afghanistan,AF,7210,0.01,433.00
2,Afghanistan,AF,73049000,0.35,"3,411.00"
3,Afghanistan,AF,7308,24.73,"41,197.00"
4,Afghanistan,AF,7309,0.55,"8,665.00"


In [9]:
# ── Join trade flows to CBAM defaults (left join from defaults) ───────────────
# Left join ensures all 119 CBAM countries are retained.
# Countries with no trade flow match receive zeros for tonnes and value.
# iso2 is filled from country_crosswalk for any countries missing it after the join.

df_merged = df_defaults_both.merge(
    df_trade[['country', 'iso2', 'cn_code', 'import_tonnes', 'eu_import_value_eur']],
    on=['country', 'cn_code'],
    how='left'
)

# Fill nulls for unmatched rows (countries with defaults but no EU trade in 2024)
df_merged['import_tonnes']      = df_merged['import_tonnes'].fillna(0)
df_merged['eu_import_value_eur'] = df_merged['eu_import_value_eur'].fillna(0)

# Fill iso2 from crosswalk for countries with no trade flow match
missing_iso2 = df_merged[df_merged['iso2'].isna()]['country'].unique()
if len(missing_iso2) > 0:
    df_crosswalk = pd.read_sql('SELECT country, iso2 FROM country_crosswalk', con)
    df_merged = df_merged.merge(df_crosswalk, on='country', how='left', suffixes=('', '_cw'))
    df_merged['iso2'] = df_merged['iso2'].fillna(df_merged['iso2_cw'])
    df_merged = df_merged.drop(columns='iso2_cw')
    print(f'iso2 filled from crosswalk for {len(missing_iso2)} countries.')

print(f'Rows after join:         {len(df_merged):,}')
print(f'Countries retained:      {df_merged["country"].nunique()}')
print(f'Zero import_tonnes rows: {(df_merged["import_tonnes"] == 0).sum():,}')

iso2 filled from crosswalk for 117 countries.
Rows after join:         10,641
Countries retained:      119
Zero import_tonnes rows: 5,754


In [10]:
# ── Core CBAM exposure calculation — embedded CO2 for both route scenarios ────
# Formula: import_tonnes x default_2026_[scenario] = embedded_co2 (tCO2)
#
# Euro costs are also computed here for internal sanity checks and the
# convenience sector pivot columns, but are NOT written to the DB as primary
# facts. The dashboard recalculates all costs at runtime from embedded_co2
# using the user-selected certificate price.

df_merged['embedded_co2_high_route'] = (
    df_merged['import_tonnes'] * df_merged['default_2026_high_route']
)
df_merged['embedded_co2_low_route'] = (
    df_merged['import_tonnes'] * df_merged['default_2026_low_route']
)

# Euro costs at base price — used only in sanity checks and sector pivot columns
df_merged['_cost_eur_high'] = df_merged['embedded_co2_high_route'] * BASE_PRICE_EUR
df_merged['_cost_eur_low']  = df_merged['embedded_co2_low_route']  * BASE_PRICE_EUR

print('Total embedded CO2 (all countries, all sectors):')
print(f'  High route: {df_merged["embedded_co2_high_route"].sum():>18,.0f} tCO2')
print(f'  Low route:  {df_merged["embedded_co2_low_route"].sum():>18,.0f} tCO2')
print()
print(f'Implied total cost at BASE_PRICE_EUR = {BASE_PRICE_EUR}:')
print(f'  High route: EUR {df_merged["_cost_eur_high"].sum():>18,.0f}')
print(f'  Low route:  EUR {df_merged["_cost_eur_low"].sum():>18,.0f}')

Total embedded CO2 (all countries, all sectors):
  High route:        206,330,628 tCO2
  Low route:         206,081,851 tCO2

Implied total cost at BASE_PRICE_EUR = 75.36:
  High route: EUR     15,549,076,127
  Low route:  EUR     15,530,328,301


In [11]:
# ── Build output table 1: exposure by country, sector, and CN code ────────────
# Grain: one row per (country, sector, cn_code).
# Most granular output table — source for all country-level aggregations.
#
# Columns stored: physical quantities and CO2 facts only.
# Euro costs are intentionally excluded: they are price-dependent and the
# dashboard recalculates them at runtime from embedded_co2 x selected price.

df_cost_by_country_sector = (
    df_merged[[
        'country', 'iso2', 'sector', 'cn_code',
        'production_route_code_high', 'production_route_high',
        'production_route_code_low',  'production_route_low',
        'has_route_variation',
        'import_tonnes', 'eu_import_value_eur',
        'default_2026_high_route', 'embedded_co2_high_route',
        'default_2026_low_route',  'embedded_co2_low_route',
    ]]
    .copy()
    .sort_values('embedded_co2_high_route', ascending=False)
    .reset_index(drop=True)
)

print(f'Output table 1 shape: {df_cost_by_country_sector.shape}')
print(f'Columns: {list(df_cost_by_country_sector.columns)}')
df_cost_by_country_sector.head(10)

Output table 1 shape: (10641, 15)
Columns: ['country', 'iso2', 'sector', 'cn_code', 'production_route_code_high', 'production_route_high', 'production_route_code_low', 'production_route_low', 'has_route_variation', 'import_tonnes', 'eu_import_value_eur', 'default_2026_high_route', 'embedded_co2_high_route', 'default_2026_low_route', 'embedded_co2_low_route']


,country,iso2,sector,cn_code,production_route_code_high,production_route_high,production_route_code_low,production_route_low,has_route_variation,import_tonnes,eu_import_value_eur,default_2026_high_route,embedded_co2_high_route,default_2026_low_route,embedded_co2_low_route
0,Russia,RU,Iron and Steel,72071210,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"3,152,317.10","1,588,901,031.00",3.53,"11,130,831.67",3.53,"11,130,831.67"
1,India,IN,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"1,584,475.51","1,040,521,909.00",4.71,"7,459,710.71",4.71,"7,459,710.71"
2,China,CN,Iron and Steel,7308,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"959,471.44","1,922,647,772.00",6.64,"6,369,451.15",6.64,"6,369,451.15"
3,Indonesia,ID,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"641,727.83","426,219,638.00",9.05,"5,809,562.09",9.05,"5,809,562.09"
4,India,IN,Iron and Steel,7210,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"1,007,722.43","878,023,485.00",4.71,"4,744,357.21",4.71,"4,744,357.21"
5,Turkey,TR,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"1,413,742.60","875,438,013.00",2.67,"3,775,064.42",2.67,"3,775,064.42"
6,China,CN,Iron and Steel,7210,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"997,240.02","914,407,145.00",3.53,"3,515,769.69",3.53,"3,515,769.69"
7,Russia,RU,Iron and Steel,7201,NaN,NaN,NaN,NaN,False,"1,029,970.84","420,882,553.00",3.34,"3,444,222.48",3.34,"3,444,222.48"
8,South Korea,KR,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"1,457,477.61","1,079,314,249.00",2.33,"3,396,394.79",2.33,"3,396,394.79"
9,Vietnam,VN,Iron and Steel,7210,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"1,219,943.14","958,902,345.00",2.61,"3,180,391.78",2.61,"3,180,391.78"


In [12]:
# ── Load global CBAM-sector exports from Comtrade ─────────────────────────────
# global_exports holds each country's total exports to all world markets
# for CBAM-relevant HS6 product codes, sourced from Comtrade.
#
# This is the correct denominator for the exposure ratio. Using EU import
# value alone would understate the denominator for large exporters like
# Russia or China, making their exposure ratio appear artificially high.
#
# Data coverage note:
# Comtrade 2024 data was not available for all CBAM countries at time of
# pull. The is_fallback_year flag marks rows where an earlier year was
# substituted. The exposure ratio (embedded_co2 / global_export_value) is
# only computed where data_year = 2024, because the numerator (embedded CO2)
# is always derived from 2024 EU trade flows. Mixing a 2024 numerator with
# a pre-2024 denominator would produce a methodologically inconsistent ratio.
# For fallback countries, exposure_ratio is set to NULL and the map shows
# no value for the '% of exports' metric. This follows standard practice
# for data journalism and consulting: a missing value with a clear
# explanation is preferable to a present value with a hidden inconsistency.
#
# global_cbam_export_value_eur and export_data_year are stored for all
# countries regardless, so the dashboard can display context values and
# surface the data year in tooltips.
#
# Join key: iso3, which is more reliable than country name strings.

# Only 2024 data is used. Fallback countries receive no join match and
# are stored with NULL for all three export columns. This is consistent
# with the 2024-only approach used for trade_flows and grid data.
# is_fallback_year is pulled so the dashboard can explain the absence.
query_global_exports = """
    SELECT
        iso3,
        data_year                 AS export_data_year,
        is_fallback_year,
        SUM(export_value_eur)     AS global_cbam_export_value_eur
    FROM global_exports
    WHERE data_year = 2024
    GROUP BY iso3, data_year, is_fallback_year
"""

df_global_exports = pd.read_sql(query_global_exports, con)

print(f'Global export countries with 2024 data: {len(df_global_exports):,}')
print(f'  Null iso3 values: {df_global_exports["iso3"].isna().sum()}')
print(f'  Countries excluded (no 2024 Comtrade data): '
      f'will receive NULL for all export columns in cbam_cost_by_country')
df_global_exports.head()

Global export countries with 2024 data: 85
  Null iso3 values: 0
  Countries excluded (no 2024 Comtrade data): will receive NULL for all export columns in cbam_cost_by_country


,iso3,export_data_year,is_fallback_year,global_cbam_export_value_eur
0,AGO,2024,0,"116,075,400.90"
1,ALB,2024,0,"371,579,052.82"
2,ARG,2024,0,"104,622,250.30"
3,ARM,2024,0,"117,856,055.02"
4,AUS,2024,0,"5,460,413,728.40"


In [13]:
# ── Build output table 2: exposure by country (all sectors aggregated) ─────────
# Grain: one row per country. All 119 CBAM countries present.
#
# Primary columns are price-independent: tonnes, CO2, and the exposure ratio.
#
# Per-sector cost pivot columns ARE included as a convenience for the dashboard
# sector donut chart, but are computed at BASE_PRICE_EUR and clearly labeled.
# See the WARNING note below.

# Total aggregation per country
df_cost_by_country = (
    df_cost_by_country_sector
    .groupby('country', as_index=False)
    .agg(
        total_import_tonnes         = ('import_tonnes',            'sum'),
        total_eu_import_value_eur   = ('eu_import_value_eur',      'sum'),
        total_embedded_co2_high     = ('embedded_co2_high_route',  'sum'),
        total_embedded_co2_low      = ('embedded_co2_low_route',   'sum'),
        has_any_route_variation     = ('has_route_variation',      'any'),
    )
    .sort_values('total_embedded_co2_high', ascending=False)
    .reset_index(drop=True)
)

# Join iso3 from crosswalk so we can join to global_exports reliably
df_crosswalk_iso = pd.read_sql(
    'SELECT country, iso2, iso3 FROM country_crosswalk', con
)
df_cost_by_country = df_cost_by_country.merge(
    df_crosswalk_iso, on='country', how='left'
)

# Join global CBAM export value, data year, and fallback flag via iso3.
# export_data_year and is_fallback_year are carried through to the DB so
# the dashboard can surface data provenance in tooltips.
df_cost_by_country = df_cost_by_country.merge(
    df_global_exports[[
        'iso3', 'global_cbam_export_value_eur',
        'export_data_year', 'is_fallback_year'
    ]],
    on='iso3', how='left'
)

# Diagnostic: countries with no Comtrade match at all
missing_global = df_cost_by_country[
    df_cost_by_country['global_cbam_export_value_eur'].isna()
]['country'].tolist()
if missing_global:
    print(f'No Comtrade match: {len(missing_global)} countries')
    print(missing_global)

# Exposure ratio: tCO2 per EUR of global CBAM-sector exports.
# Only computed where export_data_year = 2024. For fallback countries,
# the denominator is from a different year than the numerator (embedded CO2
# from 2024 EU trade flows), which would produce a methodologically
# inconsistent ratio. NULL is the correct and honest value for those rows.
# The dashboard renders no color for NULL countries on the % of exports metric
# and shows the data year in a tooltip for full transparency.
is_2024 = df_cost_by_country['export_data_year'] == 2024
denom   = df_cost_by_country['global_cbam_export_value_eur'].replace(0, float('nan'))

df_cost_by_country['cbam_eu_exposure_ratio_high'] = (
    (df_cost_by_country['total_embedded_co2_high'] / denom)
    .where(is_2024)
    .round(8)
)

df_cost_by_country['cbam_eu_exposure_ratio_low'] = (
    (df_cost_by_country['total_embedded_co2_low'] / denom)
    .where(is_2024)
    .round(8)
)

# Rank by total embedded CO2 high route (price-independent)
df_cost_by_country.insert(
    0, 'rank',
    df_cost_by_country['total_embedded_co2_high']
    .rank(ascending=False, method='min')
    .astype(int)
)

n_ratio = df_cost_by_country['cbam_eu_exposure_ratio_high'].notna().sum()
n_null  = df_cost_by_country['cbam_eu_exposure_ratio_high'].isna().sum()
print(f'Output table 2 shape (before sector pivot): {df_cost_by_country.shape}')
print(f'Exposure ratio computed: {n_ratio} countries (2024 Comtrade data)')
print(f'Exposure ratio NULL:     {n_null} countries (fallback or no Comtrade match)')

No Comtrade match: 34 countries
['Russia', 'Vietnam', 'Taiwan', 'United Arab Emirates', 'Iran', 'Belarus', 'Ghana', 'Cameroon', 'Libya', 'Venezuela', 'Tajikistan', 'Turkmenistan', 'Ethiopia', 'Iraq', 'Bangladesh', 'Gabon', 'Syria', 'Democratic Republic of the Congo', 'Cuba', 'Curacao', 'Congo', 'Eritrea', 'Equatorial Guinea', 'Rwanda', 'Eswatini', 'Haiti', 'Papua New Guinea', 'Laos', 'Sudan', 'Mongolia', 'Sierra Leone', 'Nepal', 'North Korea', 'Mali']
Output table 2 shape (before sector pivot): (119, 14)
Exposure ratio computed: 85 countries (2024 Comtrade data)
Exposure ratio NULL:     34 countries (fallback or no Comtrade match)


## WARNING: Price-Dependent Convenience Columns

The per-sector cost columns added below are computed at `BASE_PRICE_EUR = 75.36`
and stored in `cbam_cost_by_country` as a convenience for the dashboard sector
donut chart.

**These columns are scenario outputs, not stable facts.** They will become stale
if the certificate price changes. The dashboard reads them only as starting values
and should always rescale to the user-selected price at runtime:

```
cost_at_selected_price = cost_column * (selected_price / BASE_PRICE_EUR)
```

Columns affected: `cost_[sector]_high_route_eur` and `cost_[sector]_low_route_eur`

All other columns in `cbam_cost_by_country` are price-independent.

In [14]:
# ── Per-sector cost pivot columns (convenience, BASE_PRICE_EUR only) ──────────
# These columns support the dashboard sector donut chart without requiring
# an additional group-by at runtime.
#
# IMPORTANT: computed at BASE_PRICE_EUR = 75.36. The dashboard must rescale
# to the user-selected price: cost * (selected_price / BASE_PRICE_EUR).
# See the WARNING markdown cell above for details.

for scenario, cost_col in [('high_route', '_cost_eur_high'), ('low_route', '_cost_eur_low')]:
    df_pivot = (
        df_merged  # use df_merged which still holds the euro cost columns
        .groupby(['country', 'sector'])[cost_col]
        .sum()
        .unstack(fill_value=0)
        .reset_index()
    )
    df_pivot.columns = [
        f'cost_{c.lower().replace(" ", "_")}_{scenario}_eur' if c != 'country' else 'country'
        for c in df_pivot.columns
    ]
    df_cost_by_country = df_cost_by_country.merge(df_pivot, on='country', how='left')

print(f'Output table 2 final shape: {df_cost_by_country.shape}')
print(f'Columns: {list(df_cost_by_country.columns)}')
df_cost_by_country.head(5)

Output table 2 final shape: (119, 24)
Columns: ['rank', 'country', 'total_import_tonnes', 'total_eu_import_value_eur', 'total_embedded_co2_high', 'total_embedded_co2_low', 'has_any_route_variation', 'iso2', 'iso3', 'global_cbam_export_value_eur', 'export_data_year', 'is_fallback_year', 'cbam_eu_exposure_ratio_high', 'cbam_eu_exposure_ratio_low', 'cost_aluminium_high_route_eur', 'cost_cement_high_route_eur', 'cost_fertilizers_high_route_eur', 'cost_hydrogen_high_route_eur', 'cost_iron_and_steel_high_route_eur', 'cost_aluminium_low_route_eur', 'cost_cement_low_route_eur', 'cost_fertilizers_low_route_eur', 'cost_hydrogen_low_route_eur', 'cost_iron_and_steel_low_route_eur']


,rank,country,total_import_tonnes,total_eu_import_value_eur,total_embedded_co2_high,total_embedded_co2_low,has_any_route_variation,iso2,iso3,global_cbam_export_value_eur,...,cost_aluminium_high_route_eur,cost_cement_high_route_eur,cost_fertilizers_high_route_eur,cost_hydrogen_high_route_eur,cost_iron_and_steel_high_route_eur,cost_aluminium_low_route_eur,cost_cement_low_route_eur,cost_fertilizers_low_route_eur,cost_hydrogen_low_route_eur,cost_iron_and_steel_low_route_eur
0,1,China,"8,095,975.83","15,736,927,377.00","36,573,116.70","36,571,510.94",True,CN,CHN,"173,201,991,573.81",...,"308,151,498.35","5,475,440.14","58,292,961.74",15.46,"2,384,230,159.19","308,151,498.35","5,354,429.50","58,292,961.74",15.46,"2,384,230,159.19"
1,2,Turkey,"7,434,799.28","9,835,825,490.00","23,806,323.39","23,806,323.39",False,TR,TUR,"22,424,268,207.21",...,"171,467,129.81","6,155,300.36","62,404,197.90",8.97,"1,554,017,893.98","171,467,129.81","6,155,300.36","62,404,197.90",8.97,"1,554,017,893.98"
2,3,India,"4,899,194.04","5,845,842,868.00","22,833,857.84","22,833,857.78",True,IN,IND,"23,148,669,123.46",...,"53,368,164.15","59,403.07","173,495.04",0.00,"1,667,158,464.91","53,368,164.15","59,398.10","173,495.04",0.00,"1,667,158,464.91"
3,4,Russia,"8,951,307.94","4,612,590,851.00","22,509,783.47","22,509,783.47",False,RU,RUS,NaN,...,"60,797,124.40",0.00,"319,152,347.10",0.00,"1,316,387,810.59","60,797,124.40",0.00,"319,152,347.10",0.00,"1,316,387,810.59"
4,5,Ukraine,"11,546,697.32","3,287,557,505.00","12,614,285.90","12,614,285.90",False,UA,UKR,"4,883,014,002.61",...,"939,133.19","188,129,221.28","3,529,407.10",0.00,"758,014,823.71","939,133.19","188,129,221.28","3,529,407.10",0.00,"758,014,823.71"


In [15]:
# ── Build output table 3: exposure by sector (all countries aggregated) ────────
# Grain: one row per sector. Physical quantities and CO2 totals only.
# Euro costs are excluded for the same reason as the other tables:
# they are price-dependent and the dashboard handles cost calculation.

df_cost_by_sector = (
    df_cost_by_country_sector
    .groupby('sector', as_index=False)
    .agg(
        n_countries                 = ('country',               'nunique'),
        n_cn_codes                  = ('cn_code',               'nunique'),
        total_import_tonnes         = ('import_tonnes',         'sum'),
        total_eu_import_value_eur   = ('eu_import_value_eur',   'sum'),
        total_embedded_co2_high     = ('embedded_co2_high_route', 'sum'),
        total_embedded_co2_low      = ('embedded_co2_low_route',  'sum'),
    )
    .sort_values('total_embedded_co2_high', ascending=False)
    .reset_index(drop=True)
)

# CO2 share of total — price-independent equivalent of the old pct_of_total_cost
df_cost_by_sector['pct_of_total_co2_high'] = (
    df_cost_by_sector['total_embedded_co2_high']
    / df_cost_by_sector['total_embedded_co2_high'].sum()
    * 100
).round(1)

df_cost_by_sector['pct_of_total_co2_low'] = (
    df_cost_by_sector['total_embedded_co2_low']
    / df_cost_by_sector['total_embedded_co2_low'].sum()
    * 100
).round(1)

print(f'Output table 3 shape: {df_cost_by_sector.shape}')
print(df_cost_by_sector.to_string(index=False))

Output table 3 shape: (5, 9)
        sector  n_countries  n_cn_codes  total_import_tonnes  total_eu_import_value_eur  total_embedded_co2_high  total_embedded_co2_low  pct_of_total_co2_high  pct_of_total_co2_low
Iron and Steel           47         200        67,716,428.14          55,806,053,839.00           165,554,602.29          165,554,602.29                  80.20                 80.30
     Aluminium           67          28         5,881,632.10          20,054,704,690.00            16,868,806.27           16,868,806.27                   8.20                  8.20
   Fertilizers           90          27        10,151,157.74           4,451,452,006.00            14,660,623.37           14,660,623.37                   7.10                  7.10
        Cement          100           6         6,734,810.49             521,599,730.00             9,244,657.10            8,995,880.21                   4.50                  4.40
      Hydrogen           93           1               117.62 

In [16]:
# ── Sanity checks before writing to database ──────────────────────────────────
# Assertions and diagnostic prints to catch obvious problems before
# committing results. Review all output carefully.

print('=== SANITY CHECKS ===')

# 1. All 119 CBAM countries present
n_countries = df_cost_by_country['country'].nunique()
assert n_countries == 119, f'Expected 119 countries, got {n_countries}'
print(f'PASS: All 119 CBAM countries present')

# 2. No negative embedded CO2 values
assert (df_cost_by_country_sector['embedded_co2_high_route'] >= 0).all(), \
    'Negative high-route CO2 values found.'
assert (df_cost_by_country_sector['embedded_co2_low_route'] >= 0).all(), \
    'Negative low-route CO2 values found.'
print('PASS: No negative embedded CO2 values')

# 3. High route always >= low route
assert (df_cost_by_country_sector['embedded_co2_high_route']
        >= df_cost_by_country_sector['embedded_co2_low_route']).all(), \
    'Low-route CO2 exceeds high-route CO2 for some rows.'
print('PASS: High route >= low route for all rows')

# 4. No nulls in key output columns
key_cols = [
    'country', 'sector', 'import_tonnes',
    'default_2026_high_route', 'default_2026_low_route',
    'embedded_co2_high_route', 'embedded_co2_low_route'
]
nulls = df_cost_by_country_sector[key_cols].isnull().sum()
if nulls.any():
    print(f'WARNING: Nulls found in key columns:')
    print(nulls[nulls > 0])
else:
    print('PASS: No nulls in key output columns')

# 5. All five expected sectors present
expected_sectors = {'Aluminium', 'Cement', 'Fertilizers', 'Hydrogen', 'Iron and Steel'}
missing = expected_sectors - set(df_cost_by_country_sector['sector'].unique())
if missing:
    print(f'WARNING: Expected sectors missing: {missing}')
else:
    print('PASS: All five sectors present')

# 6. Countries with zero embedded CO2 (no matched trade)
zero_co2 = df_cost_by_country[
    df_cost_by_country['total_embedded_co2_high'] == 0
]['country'].tolist()
print(f'\nCountries with zero embedded CO2: {len(zero_co2)}')
print(zero_co2)

# 7. Route variation summary
n_varied = df_cost_by_country_sector['has_route_variation'].sum()
print(f'\nRows with route variation: {n_varied:,} of {len(df_cost_by_country_sector):,}')
print(f'Countries with any route variation: '
      f'{df_cost_by_country["has_any_route_variation"].sum()}')

# 8. Global export join coverage
n_matched = df_cost_by_country['global_cbam_export_value_eur'].notna().sum()
print(f'\nGlobal export value matched: {n_matched} of 119 countries')
if n_matched < 119:
    unmatched = df_cost_by_country[
        df_cost_by_country['global_cbam_export_value_eur'].isna()
    ]['country'].tolist()
    print(f'Unmatched: {unmatched}')

# 9. Top 10 countries by embedded CO2 (high route)
print('\nTop 10 countries by embedded CO2 (high route):')
print(
    df_cost_by_country[[
        'rank', 'country',
        'total_embedded_co2_high', 'total_embedded_co2_low',
        'cbam_eu_exposure_ratio_high'
    ]]
    .head(10)
    .to_string(index=False)
)

=== SANITY CHECKS ===
PASS: All 119 CBAM countries present
PASS: No negative embedded CO2 values
PASS: High route >= low route for all rows
PASS: No nulls in key output columns
PASS: All five sectors present

Countries with zero embedded CO2: 21
['Curacao', 'Brunei Darussalam', 'Congo', 'Cambodia', 'Eritrea', 'Yemen', 'Equatorial Guinea', 'Rwanda', 'Eswatini', 'Haiti', 'Papua New Guinea', 'Jamaica', 'Laos', 'Suriname', 'Sudan', 'Mongolia', 'Namibia', 'Sierra Leone', 'Nepal', 'North Korea', 'Mali']

Rows with route variation: 28 of 10,641
Countries with any route variation: 17

Global export value matched: 85 of 119 countries
Unmatched: ['Russia', 'Vietnam', 'Taiwan', 'United Arab Emirates', 'Iran', 'Belarus', 'Ghana', 'Cameroon', 'Libya', 'Venezuela', 'Tajikistan', 'Turkmenistan', 'Ethiopia', 'Iraq', 'Bangladesh', 'Gabon', 'Syria', 'Democratic Republic of the Congo', 'Cuba', 'Curacao', 'Congo', 'Eritrea', 'Equatorial Guinea', 'Rwanda', 'Eswatini', 'Haiti', 'Papua New Guinea', 'Laos', '

In [17]:
# ── Write output tables to database ──────────────────────────────────────────
# if_exists='replace' makes this notebook fully idempotent and safe to re-run.
#
# What is stored and why:
#   cbam_cost_by_country_sector: physical facts only (tonnes, CO2, defaults).
#     No euro costs. The dashboard computes costs at runtime.
#
#   cbam_cost_by_country: CO2 totals, global export value, price-independent
#     exposure ratio, plus per-sector cost pivots at BASE_PRICE_EUR as a
#     convenience (see WARNING cell above).
#
#   cbam_cost_by_sector: CO2 totals and CO2 share only. No euro costs.

tables = {
    'cbam_cost_by_country_sector': df_cost_by_country_sector,
    'cbam_cost_by_country':        df_cost_by_country,
    'cbam_cost_by_sector':         df_cost_by_sector,
}

for table_name, df in tables.items():
    df.to_sql(table_name, con, if_exists='replace', index=False)
    print(f'Written: {table_name} ({len(df):,} rows, {len(df.columns)} columns)')

print('\nAll output tables written successfully.')

Written: cbam_cost_by_country_sector (10,641 rows, 15 columns)
Written: cbam_cost_by_country (119 rows, 24 columns)
Written: cbam_cost_by_sector (5 rows, 9 columns)

All output tables written successfully.


In [18]:
# ── Verification queries ──────────────────────────────────────────────────────
# Confirm all three tables exist in the database with correct row counts.
# Also verify that no euro cost columns crept into the granular table.

print('=== DATABASE VERIFICATION ===')
for table_name, df in tables.items():
    count    = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table_name}', con).iloc[0, 0]
    expected = len(df)
    status   = 'PASS' if count == expected else 'FAIL'
    print(f'[{status}] {table_name}: {count:,} rows (expected {expected:,})')

# Confirm cbam_cost_by_country_sector contains no euro cost columns
cols_granular = pd.read_sql(
    'SELECT * FROM cbam_cost_by_country_sector LIMIT 0', con
).columns.tolist()
price_cols = [c for c in cols_granular if 'cost_eur' in c]
if price_cols:
    print(f'\nFAIL: Euro cost columns found in granular table: {price_cols}')
else:
    print('\nPASS: No euro cost columns in cbam_cost_by_country_sector')

con.close()
print('\nConnection closed. Notebook complete.')

=== DATABASE VERIFICATION ===
[PASS] cbam_cost_by_country_sector: 10,641 rows (expected 10,641)
[PASS] cbam_cost_by_country: 119 rows (expected 119)
[PASS] cbam_cost_by_sector: 5 rows (expected 5)

PASS: No euro cost columns in cbam_cost_by_country_sector

Connection closed. Notebook complete.
